# LeetCode 896 — Monotonic Array bằng Python

**Link đề:** https://leetcode.com/problems/monotonic-array/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def isMonotonic(self, nums: list[int]) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu chính xác "đơn điệu tăng" (**không giảm**, cho phép bằng) và "đơn điệu giảm" (**không tăng**).
2. Biết quy định nghĩa "với mọi `i <= j`" về điều kiện chỉ trên **các cặp kề nhau** `nums[i]`, `nums[i+1]`.
3. Đi qua 5 cách: so với mảng đã sắp xếp → hai lượt kiểm → một lượt hai cờ → một lượt xác định hướng → `all` + `zip`.
4. Nắm các cú pháp `sorted(..., reverse=True)`, `zip(nums, nums[1:])`, `all(...)`, `any(...)` và short-circuit.
5. Trace, chứng minh invariant, phân tích Time/Space, test đối chiếu, 6 bài tập.

## 1. Đề bài và ví dụ

Một mảng là **monotonic** (đơn điệu) nếu nó **tăng đơn điệu** hoặc **giảm đơn điệu**.

- Tăng đơn điệu: với mọi `i <= j`, `nums[i] <= nums[j]`.
- Giảm đơn điệu: với mọi `i <= j`, `nums[i] >= nums[j]`.

Trả `True` nếu `nums` đơn điệu.

| nums | Output | Vì sao |
|---|:---:|---|
| `[1,2,2,3]` | `true` | không giảm (cho phép 2 = 2) |
| `[6,5,4,4]` | `true` | không tăng |
| `[1,3,2]` | `false` | lên rồi xuống |

**Constraints:** `1 <= nums.length <= 10^5`, `-10^5 <= nums[i] <= 10^5`.

Lưu ý: mảng **hằng** như `[5,5,5]` vừa tăng vừa giảm đơn điệu → `true`. Mảng 1 phần tử → `true`.

## 2. Nền tảng tư duy và Python

### 2.1. Từ "mọi cặp `i <= j`" xuống "mọi cặp kề nhau"

Kiểm tra mọi cặp `(i, j)` tốn `O(n²)`. Nhưng quan hệ `<=` có tính **bắc cầu**: nếu `a <= b` và `b <= c` thì `a <= c`.

Vậy: `nums[0] <= nums[1] <= ... <= nums[n-1]` (chỉ `n − 1` cặp kề) ⇒ mọi cặp `i <= j` đều thỏa. Ngược lại, nếu mọi cặp `i <= j` thỏa thì hiển nhiên các cặp kề thỏa. → **Hai điều kiện tương đương.**

In [1]:
nums = [1, 2, 2, 3]
for i in range(len(nums) - 1):
    print(f"cặp kề ({i},{i+1}): {nums[i]} <= {nums[i+1]} ? {nums[i] <= nums[i+1]}")

cặp kề (0,1): 1 <= 2 ? True
cặp kề (1,2): 2 <= 2 ? True
cặp kề (2,3): 2 <= 3 ? True


### 2.2. `range(len(nums) - 1)` — vì sao trừ 1?

Ta đọc `nums[i + 1]`. Nếu `i` chạy tới `len(nums) - 1` thì `i + 1 = len(nums)` → `IndexError`. Nên `i` chỉ chạy tới `len(nums) - 2`, tức `range(len(nums) - 1)`. Với mảng 1 phần tử, `range(0)` rỗng → không có cặp nào → mặc nhiên đơn điệu.

In [2]:
nums = [4, 7, 9]
print("range(len(nums) - 1) =", list(range(len(nums) - 1)))
try:
    for i in range(len(nums)):
        _ = nums[i + 1]
except IndexError as e:
    print("Duyệt tới i cuối rồi đọc nums[i+1] ->", e)
print("Mảng 1 phần tử:", list(range(len([42]) - 1)), "<- không có cặp nào")

range(len(nums) - 1) = [0, 1]
Duyệt tới i cuối rồi đọc nums[i+1] -> list index out of range
Mảng 1 phần tử: [] <- không có cặp nào


### 2.3. `zip(nums, nums[1:])` — lấy các cặp kề không cần chỉ số

- `nums[1:]` là **slicing**: bản sao từ phần tử thứ 1 tới hết.
- `zip(a, b)` ghép từng cặp cùng vị trí, dừng ở dãy ngắn hơn.

Ghép `nums` với `nums[1:]` → đúng các cặp `(nums[i], nums[i+1])`.

In [3]:
nums = [6, 5, 4, 4]
print("nums[1:] =", nums[1:])
print("cặp kề  =", list(zip(nums, nums[1:])))

nums[1:] = [5, 4, 4]
cặp kề  = [(6, 5), (5, 4), (4, 4)]


### 2.4. `all(...)` và `any(...)` — short-circuit

- `all(iterable)`: `True` nếu **mọi** phần tử đúng; **dừng ngay** khi gặp phần tử sai đầu tiên.
- `any(iterable)`: `True` nếu **có ít nhất một** phần tử đúng; dừng ngay khi gặp phần tử đúng đầu tiên.
- `all([])` là `True` (không có phản ví dụ), `any([])` là `False`.

In [4]:
def noisy(x):
    print("  kiểm", x)
    return x > 0

print("all:", all(noisy(x) for x in [3, -1, 5, 7]))   # dừng ở -1
print("any:", any(noisy(x) for x in [-2, 4, -1]))     # dừng ở 4
print("all([]) =", all([]), "| any([]) =", any([]))

  kiểm 3
  kiểm -1
all: False
  kiểm -2
  kiểm 4
any: True
all([]) = True | any([]) = False


### 2.5. `sorted(nums)` và `sorted(nums, reverse=True)`

`sorted` trả list **mới** đã sắp xếp tăng dần; `reverse=True` cho giảm dần. Mảng tăng đơn điệu ⇔ nó **bằng chính bản đã sắp xếp** của nó.

In [5]:
nums = [1, 3, 2]
print(sorted(nums), sorted(nums, reverse=True), "| nums vẫn là", nums)
print([1, 2, 2, 3] == sorted([1, 2, 2, 3]))

[1, 2, 3] [3, 2, 1] | nums vẫn là [1, 3, 2]
True


## 3. Cách 1 — So với bản sắp xếp

Ý tưởng dễ nghĩ nhất: "tăng đơn điệu nghĩa là đã được sắp xếp tăng dần". Nên:

```python
return nums == sorted(nums) or nums == sorted(nums, reverse=True)
```

In [6]:
def is_monotonic_sort(nums: list[int]) -> bool:
    return nums == sorted(nums) or nums == sorted(nums, reverse=True)

for nums in ([1, 2, 2, 3], [6, 5, 4, 4], [1, 3, 2], [5]):
    print(nums, "->", is_monotonic_sort(nums))

[1, 2, 2, 3] -> True
[6, 5, 4, 4] -> True
[1, 3, 2] -> False
[5] -> True


### 3.1. Nút thắt
- Thời gian `O(n log n)` (sắp xếp) và bộ nhớ `O(n)` (bản sao) — trong khi chỉ cần **so các cặp kề**, việc `O(n)`.
- Phải sắp xếp hai lần (hoặc một lần rồi đảo) dù mảng có thể "hỏng" ngay ở cặp đầu tiên.

**Insight:** mục 2.1 cho biết chỉ cần duyệt các cặp kề → `O(n)`, `O(1)` bộ nhớ, và còn **dừng sớm** được.

## 4. Cách 2 — Hai lượt: kiểm tăng, rồi kiểm giảm

Viết hai hàm nhỏ: `is_non_decreasing` và `is_non_increasing`, mỗi hàm duyệt cặp kề và trả `False` ngay khi gặp cặp vi phạm.

In [7]:
def is_non_decreasing(nums: list[int]) -> bool:
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:      # vi phạm "không giảm"
            return False
    return True

def is_non_increasing(nums: list[int]) -> bool:
    for i in range(len(nums) - 1):
        if nums[i] < nums[i + 1]:      # vi phạm "không tăng"
            return False
    return True

def is_monotonic_two_pass(nums: list[int]) -> bool:
    return is_non_decreasing(nums) or is_non_increasing(nums)

for nums in ([1, 2, 2, 3], [6, 5, 4, 4], [1, 3, 2], [5]):
    print(nums, "->", is_monotonic_two_pass(nums))

[1, 2, 2, 3] -> True
[6, 5, 4, 4] -> True
[1, 3, 2] -> False
[5] -> True


`or` cũng short-circuit: nếu mảng đã không giảm thì **không** chạy lượt kiểm thứ hai.

## 5. Cách 3 — Lời giải chính: một lượt, hai cờ

Gộp hai lượt thành một: giữ hai biến boolean (**cờ** — flag):
- `inc = True`: "đến giờ vẫn có thể là không giảm".
- `dec = True`: "đến giờ vẫn có thể là không tăng".

Với mỗi cặp kề `(a, b)`:
- `a > b` → **không thể** là không giảm → `inc = False`.
- `a < b` → **không thể** là không tăng → `dec = False`.
- `a == b` → không ảnh hưởng gì.

Cuối cùng trả `inc or dec`. Có thể **dừng sớm** khi cả hai cờ đều `False`.

In [8]:
def is_monotonic_flags(nums: list[int]) -> bool:
    inc = dec = True
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            inc = False
        elif nums[i] < nums[i + 1]:
            dec = False
        if not inc and not dec:        # dừng sớm: đã chắc chắn không đơn điệu
            return False
    return inc or dec

for nums in ([1, 2, 2, 3], [6, 5, 4, 4], [1, 3, 2], [5], [7, 7, 7]):
    print(nums, "->", is_monotonic_flags(nums))

[1, 2, 2, 3] -> True
[6, 5, 4, 4] -> True
[1, 3, 2] -> False
[5] -> True
[7, 7, 7] -> True


## 6. Cách 4 — Một lượt, xác định hướng từ cặp **khác nhau đầu tiên**

Ý tưởng khác: chiều của mảng được quyết định bởi **cặp kề khác nhau đầu tiên**. Ghi nhớ hướng đó (`+1` tăng, `-1` giảm) và yêu cầu mọi cặp khác nhau sau đó **cùng hướng**.

So sánh hướng bằng hàm dấu: `sign(b - a)` ∈ {−1, 0, 1}. Python không có `sign` sẵn cho int; ta dùng mẹo `(b > a) - (b < a)` (vì `True - False = 1`).

In [9]:
for a, c in [(1, 5), (5, 1), (3, 3)]:
    print(f"sign({c} - {a}) = {(c > a) - (c < a)}")

sign(5 - 1) = 1
sign(1 - 5) = -1
sign(3 - 3) = 0


In [10]:
def is_monotonic_direction(nums: list[int]) -> bool:
    direction = 0                         # 0 = chưa biết hướng
    for i in range(len(nums) - 1):
        s = (nums[i + 1] > nums[i]) - (nums[i + 1] < nums[i])
        if s == 0:
            continue                      # cặp bằng nhau: không quyết định gì
        if direction == 0:
            direction = s                 # cặp khác nhau đầu tiên chốt hướng
        elif s != direction:
            return False                  # đổi hướng -> không đơn điệu
    return True

for nums in ([1, 2, 2, 3], [6, 5, 4, 4], [1, 3, 2], [2, 2, 1, 1, 3]):
    print(nums, "->", is_monotonic_direction(nums))

[1, 2, 2, 3] -> True
[6, 5, 4, 4] -> True
[1, 3, 2] -> False
[2, 2, 1, 1, 3] -> False


## 7. Cách 5 — Pythonic: `all` + `zip`

Ghép mục 2.3 và 2.4 thành một dòng:

In [11]:
def is_monotonic_pythonic(nums: list[int]) -> bool:
    pairs = list(zip(nums, nums[1:]))
    return all(a <= b for a, b in pairs) or all(a >= b for a, b in pairs)

for nums in ([1, 2, 2, 3], [6, 5, 4, 4], [1, 3, 2]):
    print(nums, "->", is_monotonic_pythonic(nums))

[1, 2, 2, 3] -> True
[6, 5, 4, 4] -> True
[1, 3, 2] -> False


Vì sao `pairs = list(...)`? `zip` trả về một **iterator** — chỉ duyệt được **một lần**. Nếu viết `p = zip(...)` rồi dùng `p` cho cả hai `all`, lượt thứ hai sẽ thấy iterator **đã cạn** → `all` của dãy rỗng = `True` → sai!

In [12]:
nums = [1, 3, 2]
p = zip(nums, nums[1:])
first = all(a <= b for a, b in p)       # dừng ở (3,2) nhưng đã tiêu thụ iterator
second = all(a >= b for a, b in p)      # iterator còn lại rỗng -> True!
print("first =", first, "| second (sai do iterator cạn) =", second, "| first or second =", first or second)

first = False | second (sai do iterator cạn) = True | first or second = True


## 8. TRACE từng bước

### 8.1. Bảng trace cách 3 (hai cờ) với `nums = [2, 2, 3, 1]`

| i | nums[i] | nums[i+1] | So sánh | inc | dec | Hành động |
|---:|---:|---:|---|:---:|:---:|---|
| — | | | khởi tạo | T | T | |
| 0 | 2 | 2 | bằng | T | T | không đổi |
| 1 | 2 | 3 | tăng | T | **F** | không thể "không tăng" |
| 2 | 3 | 1 | giảm | **F** | F | cả hai False → `return False` |

In [13]:
def trace_flags(nums):
    inc = dec = True
    print(f"nums={nums} | khởi tạo inc={inc}, dec={dec}")
    for i in range(len(nums) - 1):
        a, c = nums[i], nums[i + 1]
        if a > c:
            inc = False; rel = "giảm -> inc=False"
        elif a < c:
            dec = False; rel = "tăng -> dec=False"
        else:
            rel = "bằng -> giữ nguyên"
        print(f"  i={i}: {a} vs {c}: {rel:20} | inc={inc}, dec={dec}")
        if not inc and not dec:
            print("  Cả hai cờ False -> dừng sớm, False")
            return False
    print("  Hết mảng -> inc or dec =", inc or dec)
    return inc or dec

_ = trace_flags([2, 2, 3, 1]); print()
_ = trace_flags([6, 5, 4, 4]); print()
_ = trace_flags([7, 7, 7])

nums=[2, 2, 3, 1] | khởi tạo inc=True, dec=True
  i=0: 2 vs 2: bằng -> giữ nguyên   | inc=True, dec=True
  i=1: 2 vs 3: tăng -> dec=False    | inc=True, dec=False
  i=2: 3 vs 1: giảm -> inc=False    | inc=False, dec=False
  Cả hai cờ False -> dừng sớm, False

nums=[6, 5, 4, 4] | khởi tạo inc=True, dec=True
  i=0: 6 vs 5: giảm -> inc=False    | inc=False, dec=True
  i=1: 5 vs 4: giảm -> inc=False    | inc=False, dec=True
  i=2: 4 vs 4: bằng -> giữ nguyên   | inc=False, dec=True
  Hết mảng -> inc or dec = True

nums=[7, 7, 7] | khởi tạo inc=True, dec=True
  i=0: 7 vs 7: bằng -> giữ nguyên   | inc=True, dec=True
  i=1: 7 vs 7: bằng -> giữ nguyên   | inc=True, dec=True
  Hết mảng -> inc or dec = True


### 8.2. Trace cách 4 (hướng) — thấy cặp bằng nhau bị bỏ qua

In [14]:
def trace_direction(nums):
    direction = 0
    print("nums =", nums)
    for i in range(len(nums) - 1):
        s = (nums[i + 1] > nums[i]) - (nums[i + 1] < nums[i])
        if s == 0:
            print(f"  i={i}: {nums[i]}->{nums[i+1]} bằng nhau, bỏ qua (direction={direction})")
            continue
        if direction == 0:
            direction = s
            print(f"  i={i}: {nums[i]}->{nums[i+1]} cặp khác đầu tiên, chốt direction={direction:+d}")
        elif s != direction:
            print(f"  i={i}: {nums[i]}->{nums[i+1]} hướng {s:+d} ≠ {direction:+d} -> False")
            return False
        else:
            print(f"  i={i}: {nums[i]}->{nums[i+1]} cùng hướng {s:+d}")
    print("  -> True")
    return True

_ = trace_direction([2, 2, 1, 1, 3])

nums = [2, 2, 1, 1, 3]
  i=0: 2->2 bằng nhau, bỏ qua (direction=0)
  i=1: 2->1 cặp khác đầu tiên, chốt direction=-1
  i=2: 1->1 bằng nhau, bỏ qua (direction=-1)
  i=3: 1->3 hướng +1 ≠ -1 -> False


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def isMonotonic(self, nums: list[int]) -> bool:
        inc = dec = True
        for i in range(len(nums) - 1):
            if nums[i] > nums[i + 1]:
                inc = False
            elif nums[i] < nums[i + 1]:
                dec = False
            if not inc and not dec:
                return False
        return True            # ít nhất một cờ còn True

sol = Solution()
print(sol.isMonotonic([1, 2, 2, 3]), sol.isMonotonic([6, 5, 4, 4]), sol.isMonotonic([1, 3, 2]))

True True False


Vì sao cuối có thể `return True`? Nếu cả hai cờ cùng `False` thì đã `return False` trong vòng lặp; ra khỏi vòng nghĩa là ít nhất một cờ còn `True`.

### 9.1. Bản nộp Pythonic

In [16]:
class SolutionPythonic:
    def isMonotonic(self, nums: list[int]) -> bool:
        return (all(a <= b for a, b in zip(nums, nums[1:])) or
                all(a >= b for a, b in zip(nums, nums[1:])))

print(SolutionPythonic().isMonotonic([1, 3, 2]), SolutionPythonic().isMonotonic([5]))

False True


Ở bản này mỗi `all` tạo `zip` **mới** nên không bị lỗi iterator cạn.

## 10. Vì sao đúng?

**Bổ đề (mục 2.1):** mảng không giảm ⇔ mọi cặp kề `nums[i] <= nums[i+1]` (nhờ bắc cầu). Tương tự cho không tăng.

**Invariant cách 3:** sau khi xét các cặp kề `0..i`:
- `inc == True` ⇔ đoạn `nums[0..i+1]` không giảm,
- `dec == True` ⇔ đoạn `nums[0..i+1]` không tăng.

Khởi tạo: đoạn 1 phần tử vừa không giảm vừa không tăng → cả hai `True` ✔. Mỗi bước, cặp mới chỉ có thể **phá** một trong hai tính chất (không bao giờ khôi phục) — đúng như cách ta cập nhật cờ. Kết thúc: mảng đơn điệu ⇔ `inc or dec`.

**Dừng sớm đúng:** khi cả hai cờ `False`, mọi phần mở rộng của đoạn cũng không thể đơn điệu (vi phạm đã nằm trong mảng) → trả `False` ngay là an toàn.

## 11. Độ phức tạp và so sánh

`n = len(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Dừng sớm? |
|---|---|---|---|:---:|
| 1. So với sorted | `nums == sorted(nums)` | `O(n log n)` | `O(n)` | không |
| 2. Hai lượt | kiểm tăng, rồi giảm | `O(n)` (≤ 2 lượt) | `O(1)` | có (mỗi lượt) |
| 3. Hai cờ | `inc`, `dec` | `O(n)` một lượt | `O(1)` | **có** |
| 4. Hướng | cặp khác đầu tiên chốt hướng | `O(n)` một lượt | `O(1)` | **có** |
| 5. `all` + `zip` | một dòng | `O(n)` | `O(n)` (slice `nums[1:]`) | có (`all`) |

Mọi cách đúng phải đọc toàn bộ mảng trong trường hợp đơn điệu → không thể dưới `O(n)`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

methods = [is_monotonic_sort, is_monotonic_two_pass, is_monotonic_flags, is_monotonic_direction,
           is_monotonic_pythonic, Solution().isMonotonic, SolutionPythonic().isMonotonic]

cases = [
    ("đề VD1", [1, 2, 2, 3], True),
    ("đề VD2", [6, 5, 4, 4], True),
    ("đề VD3", [1, 3, 2], False),
    ("1 phần tử", [5], True),
    ("hằng", [7, 7, 7, 7], True),
    ("2 phần tử giảm", [2, 1], True),
    ("bằng rồi tăng rồi giảm", [2, 2, 3, 1], False),
    ("giảm rồi tăng ở cuối", [5, 4, 3, 4], False),
    ("số âm", [-5, -3, -3, 0], True),
    ("vi phạm ở cặp đầu", [3, 1, 2, 3, 4], False),
]
for name, nums, exp in cases:
    for f in methods:
        assert f(nums) == exp, (name, f)
    check(name, is_monotonic_flags(nums), exp)

PASS  đề VD1: True
PASS  đề VD2: True
PASS  đề VD3: False
PASS  1 phần tử: True
PASS  hằng: True
PASS  2 phần tử giảm: True
PASS  bằng rồi tăng rồi giảm: False
PASS  giảm rồi tăng ở cuối: False
PASS  số âm: True
PASS  vi phạm ở cặp đầu: False


In [18]:
import random
random.seed(896)
for _ in range(2000):
    n = random.randint(1, 7)
    nums = [random.randint(-2, 2) for _ in range(n)]
    # oracle O(n^2) theo đúng định nghĩa "mọi i <= j"
    inc = all(nums[i] <= nums[j] for i in range(n) for j in range(i, n))
    dec = all(nums[i] >= nums[j] for i in range(n) for j in range(i, n))
    for f in methods:
        assert f(nums) == (inc or dec)
print("2000 mảng ngẫu nhiên: 7 cách khớp oracle theo định nghĩa gốc.")

big = list(range(10 ** 5))
print("n = 10^5 tăng dần:", Solution().isMonotonic(big), "| thêm 1 phần tử phá:", Solution().isMonotonic(big + [0]))

2000 mảng ngẫu nhiên: 7 cách khớp oracle theo định nghĩa gốc.
n = 10^5 tăng dần: True | thêm 1 phần tử phá: False


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `<` / `>` chặt — từ chối các phần tử bằng nhau
Đề cho phép bằng nhau (`[1,2,2,3]` là `true`).

In [19]:
def wrong_strict(nums):
    return all(a < b for a, b in zip(nums, nums[1:])) or all(a > b for a, b in zip(nums, nums[1:]))

print("Sai :", wrong_strict([1, 2, 2, 3]), "| Đúng:", Solution().isMonotonic([1, 2, 2, 3]))

Sai : False | Đúng: True


### Lỗi 2: Chốt hướng bằng **cặp đầu tiên** dù nó bằng nhau
`[2,2,1]`: cặp đầu bằng nhau → coi là "tăng" → gặp `2 > 1` → trả `False` sai.

In [20]:
def wrong_first_pair(nums):
    if len(nums) < 2:
        return True
    increasing = nums[0] <= nums[1]          # SAI: cặp bằng nhau bị coi là tăng
    for i in range(len(nums) - 1):
        if increasing and nums[i] > nums[i + 1]:
            return False
        if not increasing and nums[i] < nums[i + 1]:
            return False
    return True

print("Sai :", wrong_first_pair([2, 2, 1]), "| Đúng:", Solution().isMonotonic([2, 2, 1]))

Sai : False | Đúng: True


### Lỗi 3: Chỉ so phần tử đầu và cuối
`nums[0] <= nums[-1]` không nói gì về phần giữa: `[1, 5, 2, 6]` có đầu < cuối nhưng không đơn điệu.

### Lỗi 4: Dùng lại một iterator `zip` cho hai lần `all` (đã minh họa ở mục 7).

### Lỗi 5: Dùng `if ... if ...` thay `if ... elif ...` mà lại đặt `dec = False` sai điều kiện
Viết `if a >= b: inc = False` (có dấu bằng) sẽ làm mảng hằng bị coi là không tăng-không giảm → sai với `[7,7,7]`.

In [21]:
def wrong_first_last(nums):
    return nums[0] <= nums[-1] or nums[0] >= nums[-1]     # luôn True!

def wrong_ge(nums):
    inc = dec = True
    for i in range(len(nums) - 1):
        if nums[i] >= nums[i + 1]:          # SAI: có dấu bằng
            inc = False
        if nums[i] <= nums[i + 1]:
            dec = False
    return inc or dec

print("Lỗi 3:", wrong_first_last([1, 5, 2, 6]), "| Đúng:", Solution().isMonotonic([1, 5, 2, 6]))
print("Lỗi 5:", wrong_ge([7, 7, 7]), "| Đúng:", Solution().isMonotonic([7, 7, 7]))

Lỗi 3: True | Đúng: False
Lỗi 5: False | Đúng: True


## 14. Mẹo nhận dạng pattern

- Tính chất "với mọi `i <= j`" của một quan hệ **bắc cầu** (`<=`, `>=`, `==`) → chỉ cần kiểm **cặp kề**.
- Cần kiểm **hai khả năng loại trừ** trong một lượt → giữ **hai cờ boolean**, mỗi phản ví dụ tắt một cờ.
- Cặp kề trong Python → `zip(nums, nums[1:])` (hoặc `itertools.pairwise` từ Python 3.10).
- Kết luận "tồn tại / mọi" → `any` / `all` với generator để được dừng sớm.

Bài cùng họ: 941 Valid Mountain Array, 665 Non-decreasing Array, 1752 Check if Array Is Sorted and Rotated, 674 Longest Continuous Increasing Subsequence.

## 15. Biến thể

### 15.1. Tăng **nghiêm ngặt** (strictly)
### 15.2. Đếm số "lần đổi hướng" (bỏ qua cặp bằng)
### 15.3. `itertools.pairwise` (Python 3.10+)

In [22]:
import sys
from itertools import pairwise    # có từ Python 3.10

def is_strictly_increasing(nums):
    return all(a < b for a, b in pairwise(nums))

def count_direction_changes(nums):
    changes, direction = 0, 0
    for a, c in pairwise(nums):
        s = (c > a) - (c < a)
        if s == 0:
            continue
        if direction != 0 and s != direction:
            changes += 1
        direction = s
    return changes

print("Python", sys.version.split()[0])
print("strict [1,2,3]:", is_strictly_increasing([1, 2, 3]), "| strict [1,2,2]:", is_strictly_increasing([1, 2, 2]))
print("Số lần đổi hướng [1,3,2,2,5,4]:", count_direction_changes([1, 3, 2, 2, 5, 4]))   # lên-xuống-lên-xuống: 3

Python 3.13.18
strict [1,2,3]: True | strict [1,2,2]: False
Số lần đổi hướng [1,3,2,2,5,4]: 3


## 16. Bài tập tự luyện

Hàm TODO chỉ định nghĩa, không được gọi khi chạy notebook.

### Bài 1 — Tính tay
Lập bảng `inc`/`dec` cho `[3, 3, 2, 2, 1, 4]`. Dừng ở cặp nào?

### Bài 2 — Dự đoán output
```python
list(zip([1, 2, 3], [1, 2, 3][1:]))
all(x > 0 for x in [])
(5 > 3) - (5 < 3)
```

### Bài 3 — Điền chỗ trống: hai cờ

In [23]:
def is_monotonic_exercise(nums) -> bool:
    inc = dec = True
    for i in range(len(nums) - 1):
        # TODO 1: nếu nums[i] > nums[i+1] -> inc = False
        # TODO 2: nếu nums[i] < nums[i+1] -> dec = False
        pass
    return inc or dec

# assert is_monotonic_exercise([1, 2, 2, 3]) is True
# assert is_monotonic_exercise([1, 3, 2]) is False

### Bài 4 — Tự viết cách "xác định hướng" (cách 4)

In [24]:
def is_monotonic_direction_exercise(nums) -> bool:
    # TODO
    pass

# assert is_monotonic_direction_exercise([2, 2, 1]) is True
# assert is_monotonic_direction_exercise([2, 2, 1, 3]) is False

### Bài 5 — Biến thể: độ dài đoạn con **liên tiếp** đơn điệu (không giảm hoặc không tăng) dài nhất

In [25]:
def longest_monotonic_run(nums) -> int:
    # TODO: giữ độ dài đoạn không giảm và không tăng kết thúc tại i
    pass

# assert longest_monotonic_run([1, 3, 2, 2, 1, 5]) == 4      # 3,2,2,1
# assert longest_monotonic_run([5]) == 1

### Bài 6 — Mini challenge: Valid Mountain Array (LeetCode 941)
Mảng là "núi" nếu dài ≥ 3, **tăng nghiêm ngặt** tới một đỉnh (không ở hai đầu) rồi **giảm nghiêm ngặt**.

In [26]:
def valid_mountain(arr) -> bool:
    # TODO: leo lên khi arr[i] < arr[i+1], rồi xuống khi arr[i] > arr[i+1]
    pass

# assert valid_mountain([0, 3, 2, 1]) is True
# assert valid_mountain([3, 5, 5]) is False
# assert valid_mountain([0, 1, 2]) is False

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | cặp bằng không đổi cờ | 3→2 tắt inc | 1→4 tắt dec → dừng ở i=4 |
| 2 | slice bỏ phần tử đầu | `all([])` | `[(1,2),(2,3)]`, `True`, `1` |
| 3 | if / elif | so sánh cặp kề | `inc = False` / `dec = False` |
| 4 | `direction = 0` | bỏ qua s == 0 | chốt lần đầu, sau đó so `s != direction` |
| 5 | hai độ dài chạy | `up = up+1 if a<=b else 1` | `best = max(best, up, down)` |
| 6 | hai vòng while | đỉnh không ở 0 hay n−1 | đi xuống phải tới đúng cuối mảng |

## 18. Đáp án tham khảo

**Bài 1:** cặp (3,3) bằng; (3,2) giảm → inc=F; (2,2) bằng; (2,1) giảm; (1,4) tăng → dec=F → cả hai F → dừng ở `i = 4`, trả `False`.

**Bài 2:** `[(1, 2), (2, 3)]`, `True`, `1`.

In [27]:
def ans_is_monotonic(nums):
    inc = dec = True
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            inc = False
        elif nums[i] < nums[i + 1]:
            dec = False
    return inc or dec

def ans_direction(nums):
    direction = 0
    for i in range(len(nums) - 1):
        s = (nums[i + 1] > nums[i]) - (nums[i + 1] < nums[i])
        if s == 0:
            continue
        if direction == 0:
            direction = s
        elif s != direction:
            return False
    return True

def ans_longest_monotonic_run(nums):
    best = up = down = 1
    for i in range(1, len(nums)):
        up = up + 1 if nums[i - 1] <= nums[i] else 1
        down = down + 1 if nums[i - 1] >= nums[i] else 1
        best = max(best, up, down)
    return best

def ans_valid_mountain(arr):
    n, i = len(arr), 0
    while i + 1 < n and arr[i] < arr[i + 1]:
        i += 1
    if i == 0 or i == n - 1:
        return False
    while i + 1 < n and arr[i] > arr[i + 1]:
        i += 1
    return i == n - 1

assert not ans_is_monotonic([3, 3, 2, 2, 1, 4])
assert list(zip([1, 2, 3], [1, 2, 3][1:])) == [(1, 2), (2, 3)] and all(x > 0 for x in []) and (5 > 3) - (5 < 3) == 1
assert ans_is_monotonic([1, 2, 2, 3]) is True and ans_is_monotonic([1, 3, 2]) is False
assert ans_direction([2, 2, 1]) is True and ans_direction([2, 2, 1, 3]) is False
assert ans_longest_monotonic_run([1, 3, 2, 2, 1, 5]) == 4 and ans_longest_monotonic_run([5]) == 1
assert ans_valid_mountain([0, 3, 2, 1]) is True
assert ans_valid_mountain([3, 5, 5]) is False
assert ans_valid_mountain([0, 1, 2]) is False
assert ans_valid_mountain([2, 1]) is False
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh và checklist submit

```python
class Solution:
    def isMonotonic(self, nums):
        inc = dec = True                       # hai giả thuyết còn sống
        for i in range(len(nums) - 1):         # chỉ xét cặp kề
            if nums[i] > nums[i + 1]:
                inc = False                    # phá "không giảm"
            elif nums[i] < nums[i + 1]:
                dec = False                    # phá "không tăng"
        return inc or dec
```

**Câu nhớ:** *"Bắc cầu → chỉ xét cặp kề; hai giả thuyết → hai cờ."*

**Checklist:**
- [ ] Cho phép phần tử **bằng nhau** (`<=`, `>=`, không dùng `<`, `>` chặt)?
- [ ] `range(len(nums) - 1)` để không tràn `nums[i+1]`?
- [ ] Mảng 1 phần tử và mảng hằng trả `True`?
- [ ] Không chốt hướng bằng cặp **bằng nhau**?
- [ ] Không dùng lại iterator `zip` đã cạn?